In [1]:
%pip install flask flask-cors numpy pandas

In [2]:
from flask import Flask, jsonify, request
from flask_cors import CORS
import numpy as np
import pandas as pd

print("Libraries imported successfully!")

Libraries imported successfully!


In [3]:
app = Flask(__name__)

CORS(app)

print("MARS-X Flask application created!")

MARS-X Flask application created!


In [4]:
@app.route("/")
def home():
    return jsonify({
        "project": "MARS-X",
        "message": "Backend is running"
    })

In [5]:
@app.route("/api/health")
def health():
    return jsonify({
        "project": "MARS-X",
        "status": "online",
        "message": "Backend is working successfully"
    })

In [6]:
@app.route("/api/geology", methods=["POST"])
def geology():

    data = request.get_json(silent=True)

    if not data:
        return jsonify({
            "status": "error",
            "message": "JSON data is required"
        }), 400

    required_fields = [
        "region_name",
        "terrain_type",
        "latitude",
        "longitude"
    ]

    for field in required_fields:
        if field not in data:
            return jsonify({
                "status": "error",
                "message": f"Missing field: {field}"
            }), 400

    try:
        latitude = float(data["latitude"])
        longitude = float(data["longitude"])
    except (ValueError, TypeError):
        return jsonify({
            "status": "error",
            "message": "Coordinates must be numbers"
        }), 400

    if not -90 <= latitude <= 90:
        return jsonify({
            "status": "error",
            "message": "Invalid latitude"
        }), 400

    if not -180 <= longitude <= 180:
        return jsonify({
            "status": "error",
            "message": "Invalid longitude"
        }), 400

    return jsonify({
        "project": "MARS-X",
        "status": "success",
        "message": "Geological data received",
        "data": {
            "region_name": data["region_name"],
            "terrain_type": data["terrain_type"],
            "latitude": latitude,
            "longitude": longitude
        }
    }), 200

In [7]:
with app.test_client() as client:

    response = client.get("/")

    print("Status Code:", response.status_code)
    print("Response:", response.json)

Status Code: 200
Response: {'message': 'Backend is running', 'project': 'MARS-X'}


In [8]:
with app.test_client() as client:

    response = client.get("/api/health")

    print("Status Code:", response.status_code)
    print("Response:", response.json)


Status Code: 200
Response: {'message': 'Backend is working successfully', 'project': 'MARS-X', 'status': 'online'}


In [9]:
test_data = {
    "region_name": "Mars Region 1",
    "terrain_type": "Crater",
    "latitude": -15.5,
    "longitude": 120.3
}

with app.test_client() as client:

    response = client.post(
        "/api/geology",
        json=test_data
    )

    print("Status Code:", response.status_code)
    print("Response:", response.json)

Status Code: 200
Response: {'data': {'latitude': -15.5, 'longitude': 120.3, 'region_name': 'Mars Region 1', 'terrain_type': 'Crater'}, 'message': 'Geological data received', 'project': 'MARS-X', 'status': 'success'}
